# 📝 과제 03: 다음 날에도 준비를 이어가는 AI 출장 비서를 만듭니다

출장 상담과 계획 승인 대기를 저장하고, 다음 실행에서 이어갑니다.

<img src="images/assignment03_overview.png" width="1050" alt="출장 AI 비서의 입력, 작업 노드, 분기와 저장 흐름">

**흐름:** 상담 State·노드·엣지 → 두 상담 저장 → 재시작·후속 상담 → 승인 State·노드·엣지 → 계획 작성·승인 대기 저장 → 재시작·승인 또는 반려.

| 단계 | 문항 | 실행 시점 |
|---|---|---|
| 상담 State·노드·엣지 | 1~3 | 정의만 준비 |
| 상담 저장 | 4 | A·B 첫 질문, 모델 각 1회 |
| 상담 복원 | 5~6 | 첫 재시작, 조회 후 A 후속 질문 1회 |
| 승인 State·노드·엣지 | 7~10 | 정의만 준비 |
| 계획 저장 | 11 | A·B 계획 생성, 모델 각 1회 |
| 승인 복원 | 12~13 | 두 번째 재시작, 조회 후 사람의 결정 반영 |

**작성 방식:** 1·5·7·9·12번은 전체 작성, 나머지는 골격 완성입니다. 기본 진행의 모델 호출은 총 5회이며 조회·승인·반려는 호출하지 않습니다.

교통·숙박 후보와 금액은 모두 학습용 가상 자료입니다. 비서는 제공 자료와 대화를 근거로 계획을 만들며, 실제 예약은 하지 않습니다. 프롬프트와 출력 스키마에는 특정 도시·날짜·예산을 넣지 않습니다.

## 준비 A: 경로·모델·출장 자료를 준비합니다

재시작 후에도 아래 두 준비 셀을 다시 실행할 수 있습니다. 이 셀들은 모델을 호출하거나 기존 대화를 변경하지 않습니다. 학생과 정답은 출력 폴더가 분리됩니다.

처음부터 새로 연습하려면 아래 **두 상담 ID와 두 승인 ID의 v1을 모두 v2로 바꾼 뒤 1번부터** 진행하세요. 저장된 기록을 복원하는 중에는 ID와 파일 경로를 유지합니다.

**공통 작성 규칙:** 노드는 입력 State와 그 안의 목록을 직접 변경하지 않고, 갱신할 필드만 반환합니다. 호출 입력은 위치 인자와 키워드 인자 어느 쪽으로 전달해도 됩니다.

**자가평가:** 각 문항을 작성한 뒤 `assert` 셀을 실행하세요. 노드 정의는 제공된 고정 응답으로 입출력을 검사하고, 실행 문항은 실제 실행 결과를 검사합니다. 자가평가 셀 자체는 GPT를 추가 호출하지 않습니다. 통과 후에도 생성된 글의 사실과 표현은 원자료와 직접 대조하세요.

.env와 모델을 준비하는 제공 셀입니다. 그대로 실행하세요. 이 단계에서는 모델을 호출하지 않습니다.

In [ ]:
# 교안과 같은 환경 설정과 모델을 사용합니다. 이 셀은 모델을 호출하지 않습니다.
import os
import json
from pathlib import Path
from typing import Annotated
from typing_extensions import TypedDict
# 자가평가는 고정 응답으로 노드의 입출력을 검사하며 추가 모델 호출을 하지 않습니다.
from types import SimpleNamespace
from unittest.mock import Mock, patch
from dotenv import find_dotenv, load_dotenv
from IPython.display import Markdown, display
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import AnyMessage, HumanMessage
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.types import Command, interrupt

material_dir = Path(".")
data_dir = material_dir / "data"
output_dir = material_dir / "output"
output_dir.mkdir(parents=True, exist_ok=True)
load_dotenv(find_dotenv(usecwd=True))
load_dotenv(material_dir.resolve().parent / ".env")
load_dotenv(material_dir / ".env", override=True)
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY를 .env에 설정한 뒤 진행하세요.")
llm = ChatOpenAI(
    model=os.getenv("OPENAI_CHAT_MODEL", "gpt-5.6-luna"),
    use_responses_api=True,
)
print("사용 모델:", llm.model_name)

두 출장 자료와 저장 경로를 확인하세요. 복원 중에는 아래 파일 경로와 네 작업 ID를 유지합니다.

In [ ]:
# 작업을 다시 찾을 때는 같은 파일 경로와 ID를 유지합니다.
trips = json.loads((data_dir / "assignment03_trips.json").read_text(encoding="utf-8"))
chat_db = output_dir / "assignment03_chat.sqlite"
approval_db = output_dir / "assignment03_approval.sqlite"
trip_configs = {
    "A": {"configurable": {"thread_id": "trip-A-v1"}},
    "B": {"configurable": {"thread_id": "trip-B-v1"}},
}
approval_configs = {
    "A": {"configurable": {"thread_id": "trip-approval-A-v1"}},
    "B": {"configurable": {"thread_id": "trip-approval-B-v1"}},
}
for label, packet in trips.items():
    print(label, packet["context"], sep="\n")
print("상담 저장:", chat_db.resolve())
print("승인 저장:", approval_db.resolve())

## 1. 상담 State를 생성합니다

출장별 자료와 사용자의 요구를 함께 유지합니다.

**작성:** 전체 코드 작성

**제공:** 자료·메시지 관련 import를 제공합니다.

- `TripChatState(TypedDict)`를 작성하세요.
- context는 str, messages는 AnyMessage 리스트에 add_messages를 적용한 필드입니다.



<details><summary>힌트</summary>

```text
thread_id가 서로 다른 출장의 State를 분리합니다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

**결과 확인:** 각 출장의 context를 유지하면서 새 질문과 답변을 누적할 수 있습니다.

In [ ]:
# [출력 확인]
print("상담 필드:", TripChatState.__annotations__)

**1번 자가평가:** 아래 제공 셀을 실행하세요. 실패하면 오류 메시지에 표시된 요구사항을 확인하세요.

In [ ]:
# [자가평가] 작성한 코드와 지문의 요구사항을 대조합니다.
# State의 자료형과 메시지 누적 규칙을 확인합니다.
from typing import get_type_hints, get_origin, get_args
from copy import deepcopy
from langchain_core.messages import AIMessage
from typing_extensions import is_typeddict
assert is_typeddict(TripChatState), "TripChatState를 TypedDict로 선언하세요."
grading_fields = get_type_hints(TripChatState, include_extras=True)
assert grading_fields.get("context") is str, "context는 str이어야 합니다."
grading_messages = grading_fields.get("messages")
assert get_origin(grading_messages) is Annotated, "messages에 Annotated와 add_messages를 적용하세요."
grading_list_type = get_args(grading_messages)[0]
assert get_origin(grading_list_type) is list and get_args(grading_list_type) == (AnyMessage,), "messages는 AnyMessage 리스트입니다."
assert add_messages in get_args(grading_messages)[1:], "messages의 누적 규칙은 add_messages입니다."
print("✅ 1번: 상담 State와 메시지 누적 규칙")
print("1번 자가평가 통과")

상담 프롬프트를 준비하세요. 출장 자료와 이전 대화를 읽고 마지막 질문에 답합니다.

In [ ]:
# 제공: 특정 목적지를 가정하지 않는 상담 프롬프트입니다.
trip_chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "사용자의 출장 준비 비서입니다. 자료와 이전 대화를 읽고 마지막 질문에 간결하게 답하세요. "
     "명시한 일정·예산·선호를 유지하고 미확정 정보를 확인할 사항으로 남기세요. "
     "자료에 없는 가격이나 예약 완료를 만들지 마세요.\n출장 자료:\n{context}"),
    MessagesPlaceholder("messages"),
])

## 2. 상담 노드를 만듭니다

비서가 후속 질문의 '그 출장'을 앞 대화와 연결해야 합니다.

**작성:** 코드 골격 완성

**제공:** trip_chat_prompt와 함수 골격을 제공합니다.

- 프롬프트와 llm을 trip_chat_chain으로 연결하세요.
- `answer_trip`은 context·messages를 체인에 한 번 전달하고 새 답변 하나를 messages 리스트로 반환하세요. 입력 State는 직접 변경하지 않습니다.

**확인:** 함수 정의만 합니다. 실제 답변은 4번과 6번에서 확인합니다.

<details><summary>힌트</summary>

```text
프롬프트의 MessagesPlaceholder에 누적된 대화 목록을 전달합니다.
```

</details>

In [ ]:
# [작성] 지문의 요구사항에 맞춰 ... 부분을 완성하세요.
# 새 답변만 반환하면 add_messages가 기존 대화에 합칩니다.
trip_chat_chain = ...
def answer_trip(state: TripChatState):
    """출장 자료와 이전 대화를 읽어 새 답변을 반환합니다."""
    reply = ...
    return ...

**2번 자가평가:** 아래 제공 셀을 실행하세요. 실패하면 오류 메시지에 표시된 요구사항을 확인하세요.

In [ ]:
# [자가평가] 작성한 코드와 지문의 요구사항을 대조합니다.
# 고정 응답을 사용하므로 이 자가평가는 GPT를 호출하지 않습니다.
from copy import deepcopy
from langchain_core.messages import AIMessage
from langchain_core.runnables import Runnable
assert isinstance(trip_chat_chain, Runnable), "trip_chat_prompt와 llm을 체인으로 연결하세요."
grading_prompt_input = {"context": "별도 출장 자료", "messages": [HumanMessage(content="별도 질문")]}
with patch.object(type(llm), "invoke", return_value=AIMessage(content="연결 확인")) as grading_model:
    grading_response = trip_chat_chain.invoke(grading_prompt_input)
    assert grading_model.call_count == 1, "제공 프롬프트와 모델을 연결하세요."
    grading_call = grading_model.call_args
    grading_prompt = grading_call.args[0] if grading_call.args else grading_call.kwargs.get("input")
    assert grading_prompt.to_messages() == trip_chat_prompt.invoke(grading_prompt_input).to_messages(), "출장 자료와 대화를 모델에 전달하세요."
    assert grading_response.text == "연결 확인", "연결한 모델의 답변을 받으세요."
grading_input = {"context": "자가평가용 출장 자료", "messages": [HumanMessage(content="준비물을 알려 주세요.")]}
grading_before = deepcopy(grading_input)
grading_reply = AIMessage(content="자가평가용 답변")
grading_chain = Mock()
grading_chain.invoke.return_value = grading_reply
grading_outputs = []
with patch.dict(globals(), {"trip_chat_chain": grading_chain, "llm": Mock()}):
    grading_outputs.append(answer_trip(grading_input))
grading_result = grading_outputs[0]
assert grading_chain.invoke.call_count == 1, "체인을 한 번 호출하세요."
grading_payload = grading_chain.invoke.call_args.args[0] if grading_chain.invoke.call_args.args else grading_chain.invoke.call_args.kwargs.get("input")
assert grading_payload == {"context": grading_before["context"], "messages": grading_before["messages"]}, "context와 이전 메시지를 전달하세요."
assert grading_result == {"messages": [grading_reply]}, "새 답변 하나만 messages 리스트로 반환하세요."
assert grading_input == grading_before, "입력 State를 직접 변경하지 마세요."
print("✅ 2번: 상담 입력 전달과 새 답변 반환 (GPT 호출 없음)")
print("2번 자가평가 통과")

## 3. 상담 엣지를 연결합니다

한 질문에 한 번 답하고 다음 질문은 별도 호출로 받습니다.

**작성:** 코드 골격 완성

**제공:** 노드 등록 골격을 제공합니다.

- TripChatState로 trip_chat_builder를 만드세요.
- START → answer_trip → END를 연결하세요. 아직 compile하지 않습니다.

**확인:** 실행과 SQLite 연결 없이 그래프 정의만 준비합니다.

<details><summary>힌트</summary>

```text
체크포인터 연결은 SQLite 파일이 열려 있는 다음 문항에서 수행합니다.
```

</details>

In [ ]:
# [작성] 지문의 요구사항에 맞춰 ... 부분을 완성하세요.
# 이 단계는 연결 정의이며 SQLite 컴파일은 열린 with 안에서 합니다.
trip_chat_builder = ...
trip_chat_builder.add_node("answer_trip", answer_trip)
...

**3번 자가평가:** 아래 제공 셀을 실행하세요. 실패하면 오류 메시지에 표시된 요구사항을 확인하세요.

In [ ]:
# [자가평가] 작성한 코드와 지문의 요구사항을 대조합니다.
# 그래프 정의만 읽고 노드는 실행하지 않습니다.
assert trip_chat_builder.state_schema is TripChatState, "상담 State 스키마를 확인하세요."
assert isinstance(trip_chat_builder, StateGraph), "TripChatState로 StateGraph를 생성하세요."
grading_graph = trip_chat_builder.compile().get_graph()
grading_edges = {(grading_edge.source, grading_edge.target) for grading_edge in grading_graph.edges}
assert grading_edges == {(START, "answer_trip"), ("answer_trip", END)}, "START → answer_trip → END를 연결하세요."
print("✅ 3번: 상담 노드와 엣지")
print("3번 자가평가 통과")

## 4. 두 출장 상담을 SQLite에 저장합니다

프로그램 종료 후에도 첫 상담을 읽을 수 있어야 합니다.

**작성:** 코드 골격 완성

**제공:** 파일 연결·입력 생성·출력 코드를 제공합니다.

- 열린 checkpointer로 trip_chat_builder를 컴파일해 trip_chat_graph에 담으세요.
- 각 config의 저장 상태를 saved로 조회하세요. values가 비었을 때만 제공 first_input으로 실행합니다.
- 저장값은 있지만 next가 남고 interrupts가 없으면 오류로 끝내지 못한 작업입니다. 제공 복구 코드처럼 invoke(None, config)로 이어갑니다.
- A·B 대화를 모두 읽어 서로의 도시나 목적이 섞이지 않았는지 확인하세요.

**확인:** 첫 실행에서 출장별 메시지는 질문·답변 2개입니다. DB가 이미 있으면 현재 기록을 조회합니다.

<details><summary>힌트</summary>

```text
같은 입력을 다시 보내는 것과 저장된 기록을 읽는 것은 다릅니다.
```

</details>

In [ ]:
# [작성] 지문의 요구사항에 맞춰 ... 부분을 완성하세요.
# 같은 셀을 다시 실행해도 이미 저장된 첫 상담은 다시 보내지 않습니다.
with SqliteSaver.from_conn_string(str(chat_db)) as checkpointer:
    trip_chat_graph = ...
    for label, config in trip_configs.items():
        ...
        if ...:
            first_input = {"context": trips[label]["context"], "messages": [HumanMessage(content=trips[label]["question"], id=f"{label}-first")]}
            ...
        elif saved.next and not saved.interrupts:
            # 입력만 저장된 뒤 실패했다면 새 입력 없이 남은 답변을 이어갑니다.
            trip_chat_graph.invoke(None, config)
        current = trip_chat_graph.get_state(config)
        print("출장:", label, "/ 다음:", current.next)
        for message in current.values["messages"]:
            print(message.type)
            display(Markdown(message.text))

**4번 자가평가:** 아래 제공 셀을 실행하세요. 실패하면 오류 메시지에 표시된 요구사항을 확인하세요.

In [ ]:
# [자가평가] 작성한 코드와 지문의 요구사항을 대조합니다.
# 기존 SQLite 기록을 읽기만 하며 모델을 호출하지 않습니다.
assert chat_db.exists(), "상담 SQLite 파일이 없습니다."
with SqliteSaver.from_conn_string(str(chat_db)) as grading_checkpointer:
    grading_graph = trip_chat_builder.compile(checkpointer=grading_checkpointer)
    for grading_label, grading_config in trip_configs.items():
        grading_saved = grading_graph.get_state(grading_config)
        assert grading_saved.values, f"{grading_label}: 저장된 상담이 없습니다."
        assert not grading_saved.next and not grading_saved.interrupts, f"{grading_label}: 답변이 끝나지 않았습니다. 오류를 고친 뒤 4번을 다시 실행하세요."
        assert grading_saved.values["context"] == trips[grading_label]["context"], f"{grading_label}: 출장 자료가 섞였습니다."
        grading_messages = grading_saved.values["messages"]
        # 이후 6번까지 마친 기록으로 다시 확인하는 경우도 허용합니다.
        assert len(grading_messages) in ({2, 4} if grading_label == "A" else {2}), f"{grading_label}: 질문과 답변 개수를 확인하세요."
        assert [grading_message.type for grading_message in grading_messages] == ["human", "ai"] * (len(grading_messages) // 2), "질문 뒤에 답변이 저장돼야 합니다."
        assert grading_messages[0].content == trips[grading_label]["question"], "해당 출장의 첫 질문을 저장하세요."
print("✅ 4번: 두 상담 저장·분리·답변 완료")
print("4번 자가평가 통과")

### 첫 번째 커널 재시작을 합니다

노트북을 저장하고 **커널 재시작만** 하세요. 전체 실행은 하지 않습니다.

1. **준비 A의 두 코드 셀**을 실행합니다.
2. 작성한 **1~3번의 코드 셀**과 2번 앞의 제공 프롬프트 셀만 다시 실행합니다.
3. **4번을 건너뛰고 5번부터** 진행합니다.

다시 준비하는 것은 import·모델·경로·State·프롬프트·노드·엣지 정의입니다. 첫 질문을 재전송하지 않아야 복원을 확인할 수 있습니다.

## 5. 저장된 두 상담을 조회합니다

재시작한 비서가 기존 기록을 읽는지 먼저 확인합니다.

**작성:** 전체 코드 작성

**제공:** chat_db와 trip_configs, 앞에서 작성한 그래프 정의를 사용합니다.

- `with SqliteSaver.from_conn_string`으로 chat_db를 열고 trip_chat_builder를 같은 checkpointer로 컴파일하세요.
- trip_configs의 A·B를 조회하여 `restored_chats` 딕셔너리에 label을 키로, 스냅샷을 값으로 저장하세요.
- 이 블록에서는 invoke를 호출하지 않습니다.



<details><summary>힌트</summary>

```text
조회 결과는 State 딕셔너리 자체가 아니라 values와 next를 가진 스냅샷입니다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

**결과 확인:** A는 부산, B는 대전 자료와 첫 상담이 남아 있고 next는 빈 튜플입니다. 초기 진행 기준 두 메시지씩입니다.

In [ ]:
# [출력 확인]
for label, saved in restored_chats.items():
    print("출장:", label, "/ 다음:", saved.next)
    if not saved.values:
        raise RuntimeError("4번 저장 여부, DB 경로, thread_id를 확인하세요.")
    print("저장 자료:", saved.values["context"])
    print("메시지 수:", len(saved.values["messages"]))
    display(Markdown(saved.values["messages"][-1].text))

**5번 자가평가:** 아래 제공 셀을 실행하세요. 실패하면 오류 메시지에 표시된 요구사항을 확인하세요.

In [ ]:
# [자가평가] 작성한 코드와 지문의 요구사항을 대조합니다.
# 복원 결과를 같은 DB에서 새로 읽은 스냅샷과 비교합니다.
assert set(restored_chats) == set(trip_configs), "A·B 스냅샷을 restored_chats에 담으세요."
with SqliteSaver.from_conn_string(str(chat_db)) as grading_checkpointer:
    grading_graph = trip_chat_builder.compile(checkpointer=grading_checkpointer)
    for grading_label, grading_config in trip_configs.items():
        grading_expected = grading_graph.get_state(grading_config)
        grading_saved = restored_chats[grading_label]
        assert grading_saved.values == grading_expected.values and grading_saved.next == grading_expected.next, "State 딕셔너리 대신 get_state 스냅샷을 저장하세요."
        assert grading_saved.values.get("context") == trips[grading_label]["context"], "복원한 출장 자료를 확인하세요."
        assert not grading_saved.next and not grading_saved.interrupts, "4번 상담이 완료된 뒤 복원하세요."
        grading_messages = grading_saved.values["messages"]
        assert len(grading_messages) in ({2, 4} if grading_label == "A" else {2}), "초기 진행은 A·B 각각 두 메시지입니다."
print("✅ 5번: 두 상담의 스냅샷 복원")
print("5번 자가평가 통과")

## 6. A 출장에만 후속 조건을 전달합니다

새 대화 입력과 승인 대기 응답을 구분합니다.

**작성:** 코드 골격 완성

**제공:** 후속 질문과 중복 메시지 확인 코드를 제공합니다.

- 같은 SQLite 파일에 연결해 A 저장 상태를 조회하세요.
- 제공 복구 코드는 next가 남고 interrupts가 없는 작업을 invoke(None, config)로 이어간 뒤 상태를 다시 읽습니다.
- 아직 같은 메시지 ID가 없을 때만 messages에 새 HumanMessage 하나를 넣어 실행하세요. context와 이전 메시지는 다시 넣지 않습니다.
- B는 조회만 하세요.

**확인:** A는 4개, B는 2개 메시지입니다. A 답변이 부산역 근처 숙박 후보와 예산을 근거로 하는지 확인합니다.

<details><summary>힌트</summary>

```text
완료된 상담의 새 질문에는 입력 딕셔너리를 사용합니다. 여기서는 중단 응답을 보내지 않습니다.
```

</details>

In [ ]:
# [작성] 지문의 요구사항에 맞춰 ... 부분을 완성하세요.
# 후속 메시지 ID를 확인해 같은 셀의 재실행으로 중복 질문이 생기지 않게 합니다.
followup_question = HumanMessage(content="그 출장의 숙소는 역에서 가까운 곳으로 골라 주세요. 교통과 숙박 예산 안에서 준비하고 싶어요.", id="A-followup")
with SqliteSaver.from_conn_string(str(chat_db)) as checkpointer:
    trip_chat_graph = trip_chat_builder.compile(checkpointer=checkpointer)
    saved = ...
    if not saved.values:
        raise RuntimeError("저장된 A 상담이 없습니다. 4번과 DB 경로를 확인하세요.")
    if saved.next and not saved.interrupts:
        # 후속 질문 ID가 저장됐어도 답변이 완료되지 않았을 수 있습니다.
        trip_chat_graph.invoke(None, trip_configs["A"])
        saved = trip_chat_graph.get_state(config=trip_configs["A"])
    if not any(message.id == followup_question.id for message in saved.values["messages"]):
        ...
    for label, config in trip_configs.items():
        current = trip_chat_graph.get_state(config)
        print("출장:", label, "/ 메시지 수:", len(current.values["messages"]))
        display(Markdown(current.values["messages"][-1].text))

**6번 자가평가:** 아래 제공 셀을 실행하세요. 실패하면 오류 메시지에 표시된 요구사항을 확인하세요.

In [ ]:
# [자가평가] 작성한 코드와 지문의 요구사항을 대조합니다.
# A의 후속 질문·답변과 B의 유지 여부를 저장 기록에서 확인합니다.
with SqliteSaver.from_conn_string(str(chat_db)) as grading_checkpointer:
    grading_graph = trip_chat_builder.compile(checkpointer=grading_checkpointer)
    for grading_label, grading_config in trip_configs.items():
        grading_saved = grading_graph.get_state(grading_config)
        assert not grading_saved.next and not grading_saved.interrupts, "답변이 미완료입니다. 오류를 고치고 6번을 다시 실행하세요."
        assert grading_saved.values["context"] == trips[grading_label]["context"], "기존 출장 자료를 유지하세요."
        grading_messages = grading_saved.values["messages"]
        assert len(grading_messages) == (4 if grading_label == "A" else 2), "A는 네 메시지, B는 두 메시지여야 합니다."
        assert [grading_message.type for grading_message in grading_messages] == ["human", "ai"] * (len(grading_messages) // 2), "질문과 답변을 함께 완료하세요."
        assert grading_messages[0].content == trips[grading_label]["question"], "첫 상담을 유지하세요."
        if grading_label == "A":
            assert grading_messages[2].id == followup_question.id and grading_messages[2].content == followup_question.content, "A에 제공된 후속 질문을 한 번만 저장하세요."
print("✅ 6번: A 후속 상담 완료와 B 유지")
print("6번 자가평가 통과")

### 출장 계획의 승인 흐름을 준비합니다

저장된 상담을 읽어 계획을 작성하고 승인 대기로 저장합니다. 상담과 승인 기록은 각각의 파일에 둡니다. 아래 7~10번은 정의만 하며 실제 계획 작성은 11번입니다.

## 7. 출장 계획 승인 State를 생성합니다

상담 기록을 근거로 계획과 미확인 사항을 검토받습니다.

**작성:** 전체 코드 작성

**제공:** 필드 이름과 역할을 제공합니다.

- `TripApprovalState(TypedDict)`를 작성하세요.
- context·conversation·plan·decision·status·approved_plan은 str입니다.
- pending_items는 list[str]이며 아직 확인할 사항입니다.
- conversation에는 저장된 메시지를 읽기 쉬운 문자열로 변환해 넣습니다.



<details><summary>힌트</summary>

```text
승인용 conversation은 이미 저장한 상담의 복사본입니다. 이 그래프에서 대화를 누적할 필요는 없습니다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

**결과 확인:** 자료와 상담 원문, 계획, 미확인 사항, 사람의 판단을 같은 State에 담을 수 있습니다.

In [ ]:
# [출력 확인]
print("승인 State:", TripApprovalState.__annotations__)

**7번 자가평가:** 아래 제공 셀을 실행하세요. 실패하면 오류 메시지에 표시된 요구사항을 확인하세요.

In [ ]:
# [자가평가] 작성한 코드와 지문의 요구사항을 대조합니다.
# 승인용 필드는 대화 누적 없이 자료형을 정의합니다.
from typing import get_type_hints
from typing_extensions import is_typeddict
assert is_typeddict(TripApprovalState), "TripApprovalState를 TypedDict로 선언하세요."
grading_fields = get_type_hints(TripApprovalState, include_extras=True)
for grading_name in ["context", "conversation", "plan", "decision", "status", "approved_plan"]:
    assert grading_fields.get(grading_name) is str, f"{grading_name}은 str입니다."
assert grading_fields.get("pending_items") == list[str], "pending_items는 list[str]입니다."
print("✅ 7번: 승인 State의 필드와 자료형")
print("7번 자가평가 통과")

계획 스키마와 프롬프트를 준비하세요. context와 conversation을 읽고 plan·pending_items로 계획과 미확인 사항을 받습니다.

In [ ]:
# 제공: 도시나 후보 수를 고정하지 않는 계획 출력 형식입니다.
from pydantic import BaseModel, Field
class PreparedPlan(BaseModel):
    plan: str = Field(description="목적, 일정, 선택한 후보 ID와 근거, 확인된 비용 소계와 예산 범위를 담은 계획")
    pending_items: list[str] = Field(description="자료나 대화만으로 확정할 수 없는 사항. 없으면 빈 목록")

trip_plan_prompt = ChatPromptTemplate.from_messages([
    ("system", "사용자의 출장 준비 비서입니다. 자료와 상담의 최신 요구를 반영해 실행 가능한 출장 계획을 작성하세요. "
     "주어진 후보 ID와 가격만 사용하세요. 고정 일정과 예산 범위를 지키고, 확인된 비용 소계와 별도 미확정 비용을 구분하세요. "
     "조건을 만족하는 후보가 없으면 대안을 지어내지 말고 확인할 사항으로 남기세요. "
     "선택 이유를 설명하고 실제 예약이 완료됐다고 쓰지 마세요."),
    ("human", "출장 자료:\n{context}\n상담 기록:\n{conversation}"),
])

## 8. 구조화된 계획을 작성하는 노드를 만듭니다

계획 본문과 미확인 사항을 별도로 읽고 검토합니다.

**작성:** 코드 골격 완성

**제공:** PreparedPlan과 trip_plan_prompt를 제공합니다.

- PreparedPlan의 구조화 출력을 사용해 trip_plan_chain을 연결하세요. strict는 True입니다.
- `prepare_trip`은 context와 conversation을 체인에 한 번 전달하세요. 입력 State는 직접 변경하지 않습니다.
- response의 plan·pending_items를 동명 State 필드로 반환하세요.

**확인:** 출력 스키마는 어떤 출장에도 같은 형식입니다. 실제 내용의 정확성은 11번에서 자료와 대조합니다.

<details><summary>힌트</summary>

```text
출력 형식이 맞아도 가격·일정이 맞는지는 별도로 확인해야 합니다.
```

</details>

In [ ]:
# [작성] 지문의 요구사항에 맞춰 ... 부분을 완성하세요.
# 계획과 미확인 사항을 분리해 검토자가 함께 읽도록 합니다.
trip_plan_chain = ...
def prepare_trip(state: TripApprovalState):
    """자료와 저장된 상담으로 계획과 확인할 사항을 작성합니다."""
    response = ...
    return ...

**8번 자가평가:** 아래 제공 셀을 실행하세요. 실패하면 오류 메시지에 표시된 요구사항을 확인하세요.

In [ ]:
# [자가평가] 작성한 코드와 지문의 요구사항을 대조합니다.
# 구조화된 고정 응답으로 입력 전달과 반환만 검사합니다.
from langchain_core.runnables import Runnable
assert isinstance(trip_plan_chain, Runnable) and trip_plan_chain.output_schema is PreparedPlan, "PreparedPlan 구조화 출력 체인을 연결하세요."
from copy import deepcopy
grading_input = {"context": "자가평가용 자료", "conversation": "human: 이동 조건을 확인해 주세요.", "plan": "", "pending_items": [], "decision": "", "status": "승인대기", "approved_plan": ""}
grading_before = deepcopy(grading_input)
grading_response = SimpleNamespace(plan="자가평가용 계획", pending_items=["예약 가능 여부"])
grading_chain = Mock()
grading_chain.invoke.return_value = grading_response
grading_outputs = []
with patch.dict(globals(), {"trip_plan_chain": grading_chain, "llm": Mock()}):
    grading_outputs.append(prepare_trip(grading_input))
grading_result = grading_outputs[0]
assert grading_chain.invoke.call_count == 1, "계획 체인을 한 번 호출하세요."
grading_payload = grading_chain.invoke.call_args.args[0] if grading_chain.invoke.call_args.args else grading_chain.invoke.call_args.kwargs.get("input")
assert grading_payload == {"context": grading_before["context"], "conversation": grading_before["conversation"]}, "자료와 상담을 체인에 전달하세요."
assert grading_result == {"plan": grading_response.plan, "pending_items": grading_response.pending_items}, "plan과 pending_items를 동명 필드로 반환하세요."
assert grading_input == grading_before, "입력 State를 직접 변경하지 마세요."
print("✅ 8번: 계획 입력 전달과 구조화 응답 반환 (GPT 호출 없음)")
print("8번 자가평가 통과")

### 승인 대기 노드를 읽습니다

아래 노드는 제공 코드입니다. 계획과 미확인 사항을 함께 보여 주고 승인 또는 반려 응답을 기다립니다. 9번에서는 이 응답을 처리하는 노드를 직접 작성합니다.

In [ ]:
# 계획 본문과 미확인 사항을 모두 보여 주고 판단을 기다립니다.
def review_trip(state: TripApprovalState):
    """출장 계획에 대한 사람의 승인 또는 반려를 반환합니다."""
    request = {"stage": "trip_approval", "plan": state["plan"], "pending_items": state["pending_items"], "choices": ["승인", "반려"]}
    decision = interrupt(request)
    return {"decision": decision}

## 9. 사람의 결정으로 결과를 기록하는 노드를 작성합니다

반려한 계획은 확정 계획으로 남으면 안 됩니다.

**작성:** 전체 코드 작성

**제공:** 앞의 review_trip 완성 코드와 함수명 record_trip을 제공합니다.

- `record_trip(state: TripApprovalState)` 전체를 작성하세요.
- decision이 승인이면 status는 `계획확정`, approved_plan은 기존 plan입니다.
- 반려이면 status는 `재검토필요`, approved_plan은 빈 문자열입니다.
- 두 필드만 반환하고 입력 State를 직접 변경하거나 모델을 호출하지 않습니다.

**확인:** 원래 plan은 반려되어도 보관되지만 approved_plan은 비어 있습니다.

<details><summary>힌트</summary>

```text
초안 보관과 확정 결과를 구분합니다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

**9번 자가평가:** 아래 제공 셀을 실행하세요. 실패하면 오류 메시지에 표시된 요구사항을 확인하세요.

In [ ]:
# [자가평가] 작성한 코드와 지문의 요구사항을 대조합니다.
# 승인과 반려를 모두 확인하며 실제 모델은 사용하지 않습니다.
from copy import deepcopy
for grading_decision, grading_status, grading_approved in [("승인", "계획확정", "보관할 원래 계획"), ("반려", "재검토필요", "")]:
    grading_input = {"context": "자료", "conversation": "상담", "plan": "보관할 원래 계획", "pending_items": ["추가 확인"], "decision": grading_decision, "status": "승인대기", "approved_plan": ""}
    grading_before = deepcopy(grading_input)
    grading_llm, grading_chat, grading_plan = Mock(), Mock(), Mock()
    grading_outputs = []
    with patch.dict(globals(), {"llm": grading_llm, "trip_chat_chain": grading_chat, "trip_plan_chain": grading_plan}):
        grading_outputs.append(record_trip(grading_input))
    grading_result = grading_outputs[0]
    assert grading_result == {"status": grading_status, "approved_plan": grading_approved}, "승인·반려에 맞는 두 필드만 반환하세요."
    assert grading_input == grading_before, "원래 계획과 입력 State를 유지하세요."
    assert not grading_llm.mock_calls and not grading_chat.mock_calls and not grading_plan.mock_calls, "사람의 결정을 기록할 때 모델을 호출하지 마세요."
print("✅ 9번: 승인·반려 처리와 원래 계획 유지")
print("9번 자가평가 통과")

## 10. 계획 작성과 승인 엣지를 연결합니다

두 판단은 같은 결과 노드에서 상태만 다르게 기록합니다.

**작성:** 코드 골격 완성

**제공:** 노드 등록 코드를 제공합니다.

- TripApprovalState로 trip_approval_builder를 만드세요.
- START → prepare_trip → review_trip → record_trip → END를 연결하세요.
- SQLite 체크포인터 연결은 11번의 열린 with 안에서 합니다.

**확인:** 고정 엣지로 세 노드를 연결합니다. 반려도 결과를 기록하고 정상 종료합니다.

<details><summary>힌트</summary>

```text
다음 노드가 같고 상태 값만 다르면 조건부 엣지가 필요하지 않습니다.
```

</details>

In [ ]:
# [작성] 지문의 요구사항에 맞춰 ... 부분을 완성하세요.
# 계획 작성 뒤에 대기하고 사람의 결정 뒤에 결과를 기록합니다.
trip_approval_builder = ...
trip_approval_builder.add_node("prepare_trip", prepare_trip)
trip_approval_builder.add_node("review_trip", review_trip)
trip_approval_builder.add_node("record_trip", record_trip)
...

**10번 자가평가:** 아래 제공 셀을 실행하세요. 실패하면 오류 메시지에 표시된 요구사항을 확인하세요.

In [ ]:
# [자가평가] 작성한 코드와 지문의 요구사항을 대조합니다.
# 세 노드의 연결만 확인하고 실행하지 않습니다.
assert trip_approval_builder.state_schema is TripApprovalState, "승인 State 스키마를 확인하세요."
assert isinstance(trip_approval_builder, StateGraph), "TripApprovalState로 StateGraph를 생성하세요."
grading_graph = trip_approval_builder.compile().get_graph()
grading_edges = {(grading_edge.source, grading_edge.target) for grading_edge in grading_graph.edges}
assert grading_edges == {(START, "prepare_trip"), ("prepare_trip", "review_trip"), ("review_trip", "record_trip"), ("record_trip", END)}, "계획 작성 → 검토 → 결과 기록 순서를 확인하세요."
print("✅ 10번: 승인 그래프의 노드와 엣지")
print("10번 자가평가 통과")

### 계획 작성용 입력을 준비합니다

11번 직전에 한 번 실행하는 제공 셀입니다. 두 번째 커널 재시작 후에는 이 셀을 건너뜁니다.

In [ ]:
# 제공: 새 연결에서 읽은 상담을 승인 입력으로 전달합니다.
approval_inputs = {}
with SqliteSaver.from_conn_string(str(chat_db)) as checkpointer:
    trip_chat_graph = trip_chat_builder.compile(checkpointer=checkpointer)
    for label, config in trip_configs.items():
        saved = trip_chat_graph.get_state(config)
        if not saved.values:
            raise RuntimeError("상담 기록이 없습니다. 4~6번을 먼저 완료하세요.")
        if saved.next or saved.interrupts or saved.values["messages"][-1].type != "ai":
            raise RuntimeError("끝나지 않은 상담이 있습니다. 4번 또는 6번에서 오류를 해결하고 다시 실행하세요.")
        if label == "A" and not any(message.id == "A-followup" for message in saved.values["messages"]):
            raise RuntimeError("A의 후속 조건을 아직 저장하지 않았습니다. 6번을 완료하세요.")
        conversation = "\n".join(message.type + ": " + message.text for message in saved.values["messages"])
        approval_inputs[label] = {
            "context": saved.values["context"], "conversation": conversation,
            "plan": "", "pending_items": [], "decision": "", "status": "승인대기", "approved_plan": "",
        }

## 11. 계획을 작성하고 승인 대기로 저장합니다

담당자가 응답하기 전에 프로그램이 종료되어도 계획이 남아야 합니다.

**작성:** 코드 골격 완성

**제공:** 저장 상담에서 만든 approval_inputs와 DB 연결·출력 코드를 제공합니다.

- 열린 체크포인터로 trip_approval_builder를 컴파일하세요.
- 각 승인 config를 조회하고 values가 없을 때만 해당 label의 approval_inputs를 실행하세요.
- 저장값이 있고 next가 남지만 interrupts가 없으면 제공 복구 코드처럼 invoke(None, config)로 남은 작업을 이어갑니다.
- 대기 중인 계획과 pending_items를 읽고 후보 ID·일정·예산을 원문과 대조하세요.

**확인:** 첫 실행에서 둘 다 승인대기이며 next는 review_trip입니다. A는 역 근처 선호를 반영하고 교통·숙박 소계 210000원, B 교통비는 80000원입니다. 식비·현지 교통비는 미확인으로 남겨야 합니다.

<details><summary>힌트</summary>

```text
체크포인터는 계획뿐 아니라 사람을 기다리는 실행 위치도 저장합니다.
```

</details>

In [ ]:
# [작성] 지문의 요구사항에 맞춰 ... 부분을 완성하세요.
# 이미 저장된 승인 건을 초기 입력으로 덮어쓰지 않습니다.
with SqliteSaver.from_conn_string(str(approval_db)) as checkpointer:
    trip_approval_graph = ...
    for label, config in approval_configs.items():
        saved = trip_approval_graph.get_state(config)
        if ...:
            ...
        elif saved.next and not saved.interrupts:
            # 계획 생성이 실패한 기록은 마지막 저장 지점에서 이어갑니다.
            trip_approval_graph.invoke(None, config)
        saved = trip_approval_graph.get_state(config)
        print("출장:", label, "/ 상태:", saved.values["status"], "/ 다음:", saved.next)
        display(Markdown(saved.values["plan"]))
        print("확인할 사항:", saved.values["pending_items"])
        print("대기 요청:", [item.value for item in saved.interrupts])

**11번 자가평가:** 아래 제공 셀을 실행하세요. 실패하면 오류 메시지에 표시된 요구사항을 확인하세요.

In [ ]:
# [자가평가] 작성한 코드와 지문의 요구사항을 대조합니다.
# 모델 출력의 문구를 고정하지 않고 저장된 계획과 실제 대기를 확인합니다.
with SqliteSaver.from_conn_string(str(approval_db)) as grading_checkpointer:
    grading_graph = trip_approval_builder.compile(checkpointer=grading_checkpointer)
    for grading_label, grading_config in approval_configs.items():
        grading_saved = grading_graph.get_state(grading_config)
        assert grading_saved.values, "계획 저장을 먼저 완료하세요."
        assert grading_saved.values["context"] == approval_inputs[grading_label]["context"], "출장 자료를 유지하세요."
        assert grading_saved.values["conversation"] == approval_inputs[grading_label]["conversation"], "저장한 상담을 계획 입력으로 사용하세요."
        assert isinstance(grading_saved.values["plan"], str) and grading_saved.values["plan"].strip(), "계획 생성이 완료되지 않았습니다."
        assert isinstance(grading_saved.values["pending_items"], list) and all(isinstance(grading_item, str) for grading_item in grading_saved.values["pending_items"]), "미확인 사항은 문자열 목록입니다."
        if grading_saved.interrupts:
            assert grading_saved.next == ("review_trip",) and grading_saved.values["status"] == "승인대기", "검토 노드에서 승인 대기를 저장하세요."
            grading_request = grading_saved.interrupts[0].value
            assert grading_request["plan"] == grading_saved.values["plan"] and grading_request["pending_items"] == grading_saved.values["pending_items"], "계획과 미확인 사항을 검토자에게 보여 주세요."
        else:
            # 13번을 마친 뒤 이 셀을 다시 확인하는 경우입니다.
            assert not grading_saved.next and grading_saved.values["status"] in {"계획확정", "재검토필요"}, "저장값만 있고 승인 대기에 도달하지 못했습니다. 11번의 오류를 해결하세요."
print("✅ 11번: 계획 저장과 승인 대기 (후보·비용은 직접 대조하세요)")
print("11번 자가평가 통과")

### 두 번째 커널 재시작을 합니다

노트북을 저장하고 **커널 재시작만** 하세요. 전체 실행은 하지 않습니다.

1. **준비 A의 두 셀**을 실행합니다.
2. **7번 State → 8번 앞의 제공 스키마·프롬프트 → 8번 작성 노드 → 9번 앞의 제공 review_trip → 9번 결과 노드 → 10번 엣지 연결** 순서로 코드 셀을 실행합니다.
3. **계획 작성용 입력 준비 셀과 11번 전체를 건너뛰고 12번부터** 진행합니다.

동일한 State·노드 이름·엣지로 그래프를 준비합니다. 모델 객체·체인·함수를 정의해도 모델을 호출하지 않습니다. 12번 조회와 13번 재개에서 이미 끝난 prepare_trip은 다시 실행되지 않아야 합니다.

## 12. 저장된 계획과 승인 대기를 복원합니다

새 프로세스에서 과거 계획을 다시 만들지 않고 읽습니다.

**작성:** 전체 코드 작성

**제공:** approval_db·approval_configs와 작성한 그래프 정의를 사용합니다.

- approval_db를 새 SqliteSaver 연결로 열고 trip_approval_builder를 컴파일하세요.
- `restored_approvals` 딕셔너리에 각 label의 get_state 결과를 저장하세요.
- 이 문항에서 invoke는 호출하지 않습니다. plan·pending_items·status·next·interrupts를 확인하세요.



<details><summary>힌트</summary>

```text
next만으로 사람의 응답 대기를 판단하지 않고 interrupts도 확인합니다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

**결과 확인:** 저장한 계획과 승인대기가 복원됩니다. 두 건 모두 아직 승인하지 않았다면 interrupts가 있습니다.

In [ ]:
# [출력 확인]
for label, saved in restored_approvals.items():
    if not saved.values:
        raise RuntimeError("11번 저장 여부, DB 경로, 승인 ID를 확인하세요.")
    print("출장:", label, "/ 상태:", saved.values["status"], "/ 다음:", saved.next)
    display(Markdown(saved.values["plan"]))
    print("미확인 사항:", saved.values["pending_items"])
    print("대기:", [item.value for item in saved.interrupts])

**12번 자가평가:** 아래 제공 셀을 실행하세요. 실패하면 오류 메시지에 표시된 요구사항을 확인하세요.

In [ ]:
# [자가평가] 작성한 코드와 지문의 요구사항을 대조합니다.
# 재시작 뒤 준비한 그래프로 DB를 읽어 복원 스냅샷과 대조합니다.
assert set(restored_approvals) == set(approval_configs), "A·B를 restored_approvals에 담으세요."
with SqliteSaver.from_conn_string(str(approval_db)) as grading_checkpointer:
    grading_graph = trip_approval_builder.compile(checkpointer=grading_checkpointer)
    for grading_label, grading_config in approval_configs.items():
        grading_saved = restored_approvals[grading_label]
        grading_expected = grading_graph.get_state(grading_config)
        assert grading_saved.values == grading_expected.values and grading_saved.next == grading_expected.next and grading_saved.interrupts == grading_expected.interrupts, "get_state로 계획과 대기 정보를 함께 복원하세요."
        assert grading_saved.values.get("plan", "").strip(), "저장한 계획이 없습니다. 11번을 확인하세요."
        if grading_saved.interrupts:
            assert grading_saved.next == ("review_trip",) and grading_saved.values["status"] == "승인대기", "승인 대기의 상태와 다음 노드를 확인하세요."
        else:
            assert not grading_saved.next and grading_saved.values["status"] in {"계획확정", "재검토필요"}, "완료되지 않은 작업입니다. 11번을 확인하세요."
print("✅ 12번: 계획·상태·대기 스냅샷 복원")
print("12번 자가평가 통과")

## 13. 승인·반려를 반영하고 결과를 저장합니다

이미 완료된 작업은 재개하지 않고 조회해야 합니다.

**작성:** 코드 골격 완성

**제공:** 검토자가 내린 trip_decisions와 출력 코드를 제공합니다. 이 시나리오에서 B는 계획 생성 뒤 미팅이 취소되어 반려합니다.

- 각 승인 상태를 조회하고 interrupts가 있을 때만 같은 config에 Command의 resume로 해당 결정을 전달하세요.
- 제공 완료 검사로 next와 interrupts가 비어 있고 status가 계획확정 또는 재검토필요인지 확인한 뒤, 현재 State를 trip_results의 label에 저장하세요. 미완료 상태는 결과 파일로 저장하지 않습니다.
- 아래 저장 셀까지 다시 실행해도 GPT를 호출하거나 대기를 새로 만들지 않아야 합니다.
- 승인 결과를 읽기 전 A 계획의 근거·비용·미확인 사항을 직접 확인하세요.



<details><summary>힌트</summary>

```text
새 질문은 입력 딕셔너리로, 사람의 대기 응답은 resume로 전달합니다.
```

</details>

In [ ]:
# [작성] 지문의 요구사항에 맞춰 ... 부분을 완성하세요.
# 실습 검토자 결정: A는 계획 승인, B는 고객 미팅 취소로 반려합니다.
trip_decisions = {"A": "승인", "B": "반려"}
trip_results = {}
with SqliteSaver.from_conn_string(str(approval_db)) as checkpointer:
    trip_approval_graph = trip_approval_builder.compile(checkpointer=checkpointer)
    for label, config in approval_configs.items():
        saved = trip_approval_graph.get_state(config)
        if not saved.values:
            raise RuntimeError("저장된 승인 요청이 없습니다. 11번을 먼저 완료하세요.")
        if ...:
            ...
        current = trip_approval_graph.get_state(config)
        if current.next or current.interrupts or current.values.get("status") not in {"계획확정", "재검토필요"}:
            raise RuntimeError("아직 완료되지 않은 승인 건입니다. 11번에서 남은 작업을 이어간 뒤 다시 확인하세요.")
        ...
        print("출장:", label, "/ 판단:", current.values["decision"], "/ 상태:", current.values["status"])
        print("남은 실행:", current.next, current.interrupts)

**결과 확인:** A는 계획확정, B는 재검토필요이며 둘 다 next·interrupts가 비어 있습니다. B의 plan은 남고 approved_plan은 비어 있습니다.

In [ ]:
# [출력 확인]
# 결과 파일을 쓰기 직전에도 DB의 완료 상태를 확인합니다.
with SqliteSaver.from_conn_string(str(approval_db)) as checkpointer:
    trip_approval_graph = trip_approval_builder.compile(checkpointer=checkpointer)
    for label in trip_results:
        saved = trip_approval_graph.get_state(approval_configs[label])
        if saved.next or saved.interrupts or saved.values.get("status") not in {"계획확정", "재검토필요"}:
            raise RuntimeError("미완료 승인 건은 저장할 수 없습니다. 11번과 13번을 확인하세요.")
for label, result in trip_results.items():
    print(label, "확정 계획:")
    display(Markdown(result["approved_plan"] or "확정 계획 없음"))
    print("원래 계획 보관:", bool(result["plan"]))
    # 미확인 사항까지 보관해 승인 뒤 필요한 확인이 사라지지 않게 합니다.
    report = {key: result[key] for key in ["plan", "pending_items", "decision", "status", "approved_plan"]}
    report_path = output_dir / (approval_configs[label]["configurable"]["thread_id"] + ".json")
    report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
    print("결과 파일:", report_path)

**13번 자가평가:** 아래 제공 셀을 실행하세요. 실패하면 오류 메시지에 표시된 요구사항을 확인하세요.

In [ ]:
# [자가평가] 작성한 코드와 지문의 요구사항을 대조합니다.
# 완료된 DB 상태와 제출 JSON을 함께 확인합니다.
assert set(trip_results) == set(approval_configs), "A·B 결과를 모두 trip_results에 담으세요."
with SqliteSaver.from_conn_string(str(approval_db)) as grading_checkpointer:
    grading_graph = trip_approval_builder.compile(checkpointer=grading_checkpointer)
    for grading_label, grading_config in approval_configs.items():
        grading_saved = grading_graph.get_state(grading_config)
        grading_result = trip_results[grading_label]
        assert not grading_saved.next and not grading_saved.interrupts, "남은 실행과 대기가 없어야 완료입니다."
        assert grading_result == grading_saved.values, "최신 State를 결과로 저장하세요."
        assert grading_result["decision"] == trip_decisions[grading_label], "해당 출장의 결정을 반영하세요."
        assert grading_result["status"] == ("계획확정" if trip_decisions[grading_label] == "승인" else "재검토필요"), "결정에 맞는 처리 상태를 확인하세요."
        assert grading_result["plan"].strip(), "반려 후에도 원래 계획을 보관하세요."
        assert grading_result["approved_plan"] == (grading_result["plan"] if trip_decisions[grading_label] == "승인" else ""), "승인한 계획만 확정 계획으로 보관하세요."
        grading_path = output_dir / (grading_config["configurable"]["thread_id"] + ".json")
        assert grading_path.exists(), "출력 확인 셀을 실행해 결과 JSON을 저장하세요."
        grading_report = json.loads(grading_path.read_text(encoding="utf-8"))
        assert grading_report == {grading_key: grading_result[grading_key] for grading_key in ["plan", "pending_items", "decision", "status", "approved_plan"]}, "JSON에 계획·미확인 사항·판단·상태·확정 계획을 그대로 저장하세요."
print("✅ 13번: 승인·반려 완료와 결과 JSON")
print("13번 자가평가 통과")

## 마무리합니다

- 상담·승인 대기 각각에서 실제 커널을 재시작했는지 확인하세요.
- A 후속 대화만 늘고 B 대화는 유지되는지 확인하세요.
- 승인 대기 재개 시 계획을 다시 생성하지 않는지 확인하세요.
- 제출: 완성한 노트북, 승인·반려 결과 JSON 2개, 재시작 전후 상태 출력. SQLite 파일은 로컬 복원 확인에 사용합니다.